# NULLXES 90D ORACLE — Colab

Репозиторий: https://github.com/MagistrTheOne/NULLXES-MAGA-ORACLE

**Runtime → CPU.** GPU не используется.

Результаты являются условными исходами модели при заданных допущениях и не являются прогнозом реальных геополитических, военных или экономических событий.

Порядок: clone → вводный инпут → pytest → 1k smoke (графики + briefing) → ladder 1k/10k/100k → скачать xlsx.

## 1. Clone / update с GitHub

In [ ]:
import os, pathlib
REPO = "https://github.com/MagistrTheOne/NULLXES-MAGA-ORACLE.git"
ROOT = "/content/NULLXES-MAGA-ORACLE"
if pathlib.Path(ROOT, ".git").exists():
    %cd /content/NULLXES-MAGA-ORACLE
    !git pull --ff-only origin main
else:
    %cd /content
    !git clone {REPO} NULLXES-MAGA-ORACLE
    %cd /content/NULLXES-MAGA-ORACLE
!pip install -q -r requirements.txt
print("cwd", os.getcwd())

## 2. Вводный инпут

Правите словарь `IN` и `LIVE_TEXT`. Это и есть вход, который можно менять перед каждым прогоном.

`LIVE_TEXT`: по одному событию на строку.

`2026-09-29 GEO санкционный комплаенс остановил платёж`

Или JSONL. Пустая строка = не накладывать новости.

In [ ]:
IN = {
    "worlds": 1000,            # для smoke; ladder ниже берёт 1k/10k/100k
    "seed": 20260928,
    "burn_monthly": 200000.0,  # ASSUMPTION / PLACEHOLDER
    "bridge_capital": 0.0,
    "cmin": 0.0,
    "p_bank": 0.22,            # ASSUMPTION / PLACEHOLDER
    "p_deal": 0.12,            # ASSUMPTION / PLACEHOLDER
    "contract_daily": 0.0,
}

LIVE_TEXT = """
# вставьте события; строки с # игнорируются
# 2026-09-29 GEO санкционный комплаенс остановил платёж
""".strip()

EVENTS_URL = ""  # например ваш JSON endpoint; пусто = выкл

from pathlib import Path
live_path = Path("/content/NULLXES-MAGA-ORACLE/events/live.jsonl")
if LIVE_TEXT and not LIVE_TEXT.startswith("#"):
    live_path.write_text(LIVE_TEXT + "\n", encoding="utf-8")
    print("wrote", live_path)
else:
    print("live.jsonl unchanged (only comments / empty)")
print("IN =", IN)

## 3. Tests

In [ ]:
%cd /content/NULLXES-MAGA-ORACLE
!python -m pytest tests -q

## 4. Smoke с вашим инпутом + автографики с разбором

In [ ]:
%cd /content/NULLXES-MAGA-ORACLE
cmd = [
    "python", "run.py",
    "--config", "configs/baseline.yaml",
    "--live-input", "configs/live_input.yaml",
    "--worlds", str(IN["worlds"]),
    "--seed", str(IN["seed"]),
    "--burn", str(IN["burn_monthly"]),
    "--bridge", str(IN["bridge_capital"]),
    "--cmin", str(IN["cmin"]),
    "--p-bank", str(IN["p_bank"]),
    "--p-deal", str(IN["p_deal"]),
    "--set", f"contract.daily_inflow={IN['contract_daily']}",
    "--events", "events/live.jsonl",
    "--skip-tests",
    "--skip-solver",
]
if EVENTS_URL:
    cmd += ["--events-url", EVENTS_URL]
import subprocess, sys
print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path
root = Path("/content/NULLXES-MAGA-ORACLE")
brief = root / "outputs" / "PIZDEC_BRIEFING.md"
if brief.exists():
    display(Markdown(brief.read_text(encoding="utf-8")))
charts = [
    "situation_onepager.png",
    "cash_bands.png",
    "ttfb.png",
    "survival_policy.png",
    "c90.png",
    "ai_time.png",
]
for name in charts:
    p = root / "outputs" / "charts" / name
    if p.exists():
        display(Image(filename=str(p), width=900))

## 5. Production ladder 1k → 10k → 100k + solver + Excel

Те же `--burn / --p-bank / --events`, но лестница масштабов. На Colab CPU — минуты.

In [ ]:
%cd /content/NULLXES-MAGA-ORACLE
cmd = [
    "python", "run.py", "--ladder",
    "--config", "configs/baseline.yaml",
    "--live-input", "configs/live_input.yaml",
    "--seed", str(IN["seed"]),
    "--burn", str(IN["burn_monthly"]),
    "--bridge", str(IN["bridge_capital"]),
    "--cmin", str(IN["cmin"]),
    "--p-bank", str(IN["p_bank"]),
    "--p-deal", str(IN["p_deal"]),
    "--set", f"contract.daily_inflow={IN['contract_daily']}",
    "--events", "events/live.jsonl",
    "--skip-tests",
]
if EVENTS_URL:
    cmd += ["--events-url", EVENTS_URL]
import subprocess
subprocess.check_call(cmd)

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path
from google.colab import files
root = Path("/content/NULLXES-MAGA-ORACLE")
brief = root / "outputs" / "PIZDEC_BRIEFING.md"
if brief.exists():
    display(Markdown(brief.read_text(encoding="utf-8")))
for name in ["situation_onepager.png", "cash_bands.png", "sensitivity_tornado.png", "ai_time.png"]:
    p = root / "outputs" / "charts" / name
    if p.exists():
        display(Image(filename=str(p), width=900))
xlsx = root / "outputs" / "NULLXES_90D_ORACLE.xlsx"
print("xlsx", xlsx, "exists", xlsx.exists())
if xlsx.exists():
    files.download(str(xlsx))

## Realtime — как пользоваться

Модель **не парсит интернет сама** и не выдаёт новости за вероятности.

Рабочий контур:
1. Кидаете заголовок в `LIVE_TEXT` или в `events/live.jsonl` в репо → `git push` → в Colab `git pull`.
2. Либо свой JSON URL в `EVENTS_URL` (Google Sheet / webhook / внутренний feed).
3. Повторный прогон **с тем же seed**: меняется overlay, не кости. Delta = новость, не новый рандом.

`observed` = событие уже случилось (форсим категорию в этот день во всех мирах).
`risk_up` = поднимаем интенсивность с этого дня до горизонта.

Лексикон (ASSUMPTION): санкции/SWIFT→GEO, банк/кредит→FIN, blackout/энергия→SEC, инфляция/топливо→MACRO, GPU/сделка→BIZ.